# Eight Weeks Without the REM

*July 5 – August 17, 2025: When the REM suspended service between Brossard and Gare Centrale, the RTL put a shuttle bus network in place for the sake of service continuity. During that downtime, where did the people that relied on the REM go? What did they do instead?*

*This notebook uses RTL GTFS data and Bixi trip records to assess the shuttle system based on stop coverage, frequency, and travel times.*

> A note on "headways", for those unfamiliar: a **headway** is the **measurement of time interval between consecutive vehicles** (such as a bus or train) traveling along the same route.
> This notebook refers to them a lot.

---

## 1. Data Setup

In [24]:
import zipfile
from pathlib import Path

import pandas as pd
import gtfs_kit as gk

In [25]:
# define data paths
DATA_DIR = Path("data")
GTFS_DIR = DATA_DIR / "gtfs"
BIXI_DIR = DATA_DIR / "bixi"
REM_DIR  = DATA_DIR / "rem"

# create directories for data
for d in [GTFS_DIR, BIXI_DIR, REM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [26]:
# set up unzip function that takes a Path as input
def unzip(zip_path: Path) -> None:
    extract_dir = zip_path.with_suffix("")
    if extract_dir.exists():
        print(f"  {extract_dir.name}/ ...already extracted, skipping")
        return
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    print(f"  {zip_path.name} → {extract_dir.name}/")

# unzip RTL GTFS
print("RTL GTFS feeds:")
for zp in sorted(GTFS_DIR.glob("*.zip")):
    unzip(zp)

# unzip Bixi
print("\nBixi trip data:")
for zp in sorted(BIXI_DIR.glob("*.zip")):
    unzip(zp)

# unzip REM GTFS
print("\nREM GTFS feeds:")
for zp in sorted(REM_DIR.glob("*.zip")):
    unzip(zp)

RTL GTFS feeds:
  20250623/ ...already extracted, skipping
  20250707/ ...already extracted, skipping
  20250825/ ...already extracted, skipping

Bixi trip data:
  DonneesOuvertes2024_010203040506070809101112/ ...already extracted, skipping
  DonneesOuvertes2025_010203040506070809101112/ ...already extracted, skipping

REM GTFS feeds:
  gtfs/ ...already extracted, skipping


In [27]:
# summarize data after being unzipped

print("RTL GTFS data overview:")
for d in sorted(GTFS_DIR.iterdir()):
    if d.is_dir():
        print(f"  {d.name}: {len(list(d.iterdir()))} files")

print("\nBixi data overview:")
for d in sorted(BIXI_DIR.iterdir()):
    if d.is_dir():
        csvs = list(d.glob("*.csv"))
        print(f"  {d.name}: {len(csvs)} CSV files")

print("\nREM GTFS data overview:")
for d in sorted(REM_DIR.iterdir()):
    if d.is_dir():
        print(f"  {d.name}: {len(list(d.iterdir()))} files")

RTL GTFS data overview:
  20250623: 12 files
  20250707: 12 files
  20250825: 12 files

Bixi data overview:
  DonneesOuvertes2024_010203040506070809101112: 1 CSV files
  DonneesOuvertes2025_010203040506070809101112: 1 CSV files

REM GTFS data overview:
  gtfs: 12 files


---

## 2. The REM Baseline

Before July 5, the REM's Brossard branch connected five stations from the South Shore to downtown in under 25 minutes. This section profiles that service — stop sequence, scheduled travel time, and peak headways — to establish what riders actually lost when service suspended.

*Note: this feed reflects current REM service (June 2026). The Brossard branch has been operationally stable since 2023, so we treat it as representative of pre-closure service levels.*

In [ ]:
# load REM GTFS tables directly
REM_GTFS = Path("data/rem/gtfs")
rem_stops     = pd.read_csv(REM_GTFS / "stops.txt")
rem_trips     = pd.read_csv(REM_GTFS / "trips.txt")
rem_stop_times = pd.read_csv(REM_GTFS / "stop_times.txt")
rem_calendar  = pd.read_csv(REM_GTFS / "calendar.txt")

# Brossard branch stations, south to north
BRANCH_CODES = ["RIV", "DUQ", "PAN", "IDS", "GCT"]
BRANCH_NAMES = {
    "RIV": "Brossard",
    "DUQ": "Du Quartier",
    "PAN": "Panama",
    "IDS": "Île-des-Soeurs",
    "GCT": "Gare Centrale",
}

# platform-level stops on the branch (QUAI stops only, no entrances or turnaround zones)
branch_stops = rem_stops[
    rem_stops.stop_id.str.contains("QUAI") & # type: ignore
    rem_stops.stop_id.str.extract(r"_([A-Z]{3})$")[0].isin(BRANCH_CODES)
].copy()
branch_stops["station_code"] = branch_stops.stop_id.str.extract(r"_([A-Z]{3})$")

print(f"{len(branch_stops)} branch platform stops across {branch_stops.station_code.nunique()} stations")

11 branch platform stops across 5 stations


In [29]:
# GTFS times can exceed 24:00:00 for post-midnight service
def to_minutes(t: str) -> float:
    h, m, s = map(int, t.split(":"))
    return h * 60 + m + s / 60

# filter stop_times to branch platforms and parse times
branch_st = rem_stop_times[rem_stop_times.stop_id.isin(branch_stops.stop_id)].copy()
branch_st["station_code"] = branch_st.stop_id.str.extract(r"_([A-Z]{3})$")
branch_st["dep_min"] = branch_st.departure_time.apply(to_minutes)
branch_st["arr_min"] = branch_st.arrival_time.apply(to_minutes)

# candidates: trips serving both Brossard and Gare Centrale
trips_at_riv = set(branch_st[branch_st.station_code == "RIV"].trip_id)
trips_at_gct = set(branch_st[branch_st.station_code == "GCT"].trip_id)
candidates = branch_st[branch_st.trip_id.isin(trips_at_riv & trips_at_gct)]

# northbound trips: Brossard stop_sequence < Gare Centrale stop_sequence
riv_seq = candidates[candidates.station_code == "RIV"].groupby("trip_id")["stop_sequence"].min()
gct_seq = candidates[candidates.station_code == "GCT"].groupby("trip_id")["stop_sequence"].min()
northbound_ids = riv_seq[riv_seq < gct_seq].index

print(f"Northbound trips identified: {len(northbound_ids)}")

nb_st = branch_st[branch_st.trip_id.isin(northbound_ids)].copy()

# elapsed time from Brossard departure; origin stop is 0 by definition
riv_dep = nb_st[nb_st.station_code == "RIV"].groupby("trip_id")["dep_min"].min()
nb_st["elapsed"] = nb_st["arr_min"] - nb_st.trip_id.map(riv_dep)
nb_st.loc[nb_st.station_code == "RIV", "elapsed"] = 0.0

# median elapsed time at each station across all northbound trips
profile = (
    nb_st
    .groupby("station_code")["elapsed"]
    .median()
    .reindex(BRANCH_CODES)
    .reset_index()
    .rename(columns={"elapsed": "min_from_brossard"})
)
profile["station"] = profile.station_code.map(BRANCH_NAMES)
profile["min_from_brossard"] = profile.min_from_brossard.round(1)

print()
print(profile[["station", "min_from_brossard"]].to_string(index=False))

Northbound trips identified: 536

       station  min_from_brossard
      Brossard                0.0
   Du Quartier                2.0
        Panama                5.7
Île-des-Soeurs               10.6
 Gare Centrale               17.8


In [30]:
# weekday service IDs from calendar
weekday_service_ids = rem_calendar[rem_calendar.monday == 1].service_id
weekday_trip_ids = rem_trips[rem_trips.service_id.isin(weekday_service_ids)].trip_id

# northbound weekday departures from Brossard, sorted chronologically
brossard_deps = (
    nb_st[
        nb_st.trip_id.isin(weekday_trip_ids) &
        (nb_st.station_code == "RIV")
    ]
    .drop_duplicates(subset=["trip_id"])
    .sort_values("dep_min")
    .copy()
)
brossard_deps["headway_min"] = brossard_deps.dep_min.diff()

# all-day (5am–11pm) and morning peak (7–9am)
allday = brossard_deps[brossard_deps.dep_min.between(5 * 60, 23 * 60)]
peak   = brossard_deps[brossard_deps.dep_min.between(7 * 60,  9 * 60)]

print("Weekday headways at Brossard (northbound):")
print(f"  All-day median:        {allday.headway_min.median():.1f} min")
print(f"  Morning peak median:   {peak.headway_min.median():.1f} min")
print(f"  Morning peak min:      {peak.headway_min.min():.1f} min")

Weekday headways at Brossard (northbound):
  All-day median:        3.5 min
  Morning peak median:   3.5 min
  Morning peak min:      3.5 min


---

## 3. The Replacement Options

The RTL didn't add a new shuttle route, they leaned on an existing one. Route 722 (Panama →  Terminus Centre-Ville) is a direct express bus running the same corridor as the REM's Brossard branch. During the closure, RTL boosted it dramatically.

A second service, T72 (a collective taxi between Terminus Panama and Île-des-Soeurs), also gained trips: likely to serve Île-des-Soeurs residents stranded by their now-closed station.

This section profiles how route 722 changed during the closure and compares it to the REM baseline.

In [31]:
# load RTL GTFS tables for both periods
def load_rtl(snapshot: str) -> dict:
    path = GTFS_DIR / snapshot
    return {
        "routes":     pd.read_csv(path / "routes.txt"),
        "trips":      pd.read_csv(path / "trips.txt"),
        "stop_times": pd.read_csv(path / "stop_times.txt"),
        "stops":      pd.read_csv(path / "stops.txt"),
        "calendar":   pd.read_csv(path / "calendar.txt"),
    }

base = load_rtl("20250623")
mid  = load_rtl("20250707")

# Terminus Panama stop IDs (the REM bus terminal, not street stops)
terminus_panama_ids = base["stops"][base["stops"].stop_name == "Terminus Panama"].stop_id

print(f"Terminus Panama stop IDs: {terminus_panama_ids.tolist()}")

Terminus Panama stop IDs: [3492, 4429, 5923, 5925, 5927]


In [32]:
def profile_722(feed: dict, label: str) -> pd.Series:
    r722_id = feed["routes"][feed["routes"].route_short_name == "722"].route_id.iloc[0]
    weekday_sid = feed["calendar"][feed["calendar"].monday == 1].service_id.iloc[0]
    weekday_722 = feed["trips"][
        (feed["trips"].route_id == r722_id) &
        (feed["trips"].service_id == weekday_sid)
    ].trip_id

    # northbound: Panama is not the last stop of the trip
    max_seq = feed["stop_times"][feed["stop_times"].trip_id.isin(weekday_722)].groupby("trip_id")["stop_sequence"].max()
    panama_st = feed["stop_times"][
        feed["stop_times"].trip_id.isin(weekday_722) &
        feed["stop_times"].stop_id.isin(terminus_panama_ids)
    ].copy()
    panama_st = panama_st[panama_st.apply(lambda r: r.stop_sequence < max_seq[r.trip_id], axis=1)]
    panama_st["dep_min"] = panama_st.departure_time.apply(to_minutes)
    panama_st = panama_st.sort_values("dep_min").drop_duplicates(subset=["trip_id"])
    panama_st["headway"] = panama_st.dep_min.diff()

    allday = panama_st[panama_st.dep_min.between(5 * 60, 23 * 60)]
    peak   = panama_st[panama_st.dep_min.between(7 * 60,  9 * 60)]

    return pd.Series({
        "period":              label,
        "weekday_trips":       len(panama_st),
        "allday_headway_med":  round(allday.headway.median(), 1),
        "peak_headway_med":    round(peak.headway.median(), 1),
    })

rows = [profile_722(base, "Baseline (Jun 23)"), profile_722(mid, "Closure (Jul 7)")]
summary_722 = pd.DataFrame(rows).set_index("period")
print("Route 722: Panama northbound, weekdays")
print(summary_722.to_string())

Route 722: Panama northbound, weekdays
                   weekday_trips  allday_headway_med  peak_headway_med
period                                                                
Baseline (Jun 23)             27                10.0               NaN
Closure (Jul 7)              183                 4.0               2.0


In [39]:
# travel time: median trip duration from Terminus Panama to Terminus Centre-Ville
r722_id = mid["routes"][mid["routes"].route_short_name == "722"].route_id.iloc[0]
weekday_sid = mid["calendar"][mid["calendar"].monday == 1].service_id.iloc[0]

# direction 1 = northbound (Panama → Centre-Ville)
weekday_722_nb = mid["trips"][
    (mid["trips"].route_id == r722_id) &
    (mid["trips"].service_id == weekday_sid) &
    (mid["trips"].direction_id == 1)
].trip_id

terminus_cv_ids = mid["stops"][mid["stops"].stop_name == "Terminus Centre-Ville"].stop_id

st_722_nb = mid["stop_times"][mid["stop_times"].trip_id.isin(weekday_722_nb)].copy()
st_722_nb["dep_min"] = st_722_nb.departure_time.apply(to_minutes)
st_722_nb["arr_min"] = st_722_nb.arrival_time.apply(to_minutes)

panama_dep = st_722_nb[st_722_nb.stop_id.isin(terminus_panama_ids)].groupby("trip_id")["dep_min"].min()
cv_arr     = st_722_nb[st_722_nb.stop_id.isin(terminus_cv_ids)].groupby("trip_id")["arr_min"].max()

travel_times = (cv_arr - panama_dep).dropna()
print(f"Route 722 travel time, Panama →  Centre-Ville (weekday northbound):")
print(f"  Median: {travel_times.median():.1f} min")
print(f"  Range:  {travel_times.min():.0f}–{travel_times.max():.0f} min")
print()

# summary comparison
print("─" * 46)
print(f"{'':25} {'REM':>8}  {'722':>8}")
print("─" * 46)
print(f"{'Travel time (min)':25} {'17.8':>8}  {travel_times.median():>8.1f}")
print(f"{'All-day headway (min)':25} {'3.5':>8}  {summary_722.loc['Closure (Jul 7)', 'allday_headway_med']:>8.1f}")
print(f"{'Peak headway (min)':25} {'3.5':>8}  {summary_722.loc['Closure (Jul 7)', 'peak_headway_med']:>8.1f}")
print("─" * 46)

Route 722 travel time, Panama →  Centre-Ville (weekday northbound):
  Median: 24.0 min
  Range:  16–27 min

──────────────────────────────────────────────
                               REM       722
──────────────────────────────────────────────
Travel time (min)             17.8      24.0
All-day headway (min)          3.5       4.0
Peak headway (min)             3.5       2.0
──────────────────────────────────────────────


---

## 4. The Bixi Signal

Route 722 tells us what RTL scheduled, it says nothing about what riders actually did. If South Shore commuters couldn't take the REM to Gare Centrale, did outgoing Bixi trips near the station change?

Gare Centrale is the island-side terminus of the Brossard branch. On a typical weekday, REM riders arriving downtown might continue on foot or by Bixi. Conversely, the closure may have redirected demand away from that node entirely.

We compare Bixi trips starting or ending within 500m of Gare Centrale during peak commute hours (7–9am, 5–7pm) across the same calendar window in 2024 and 2025 (July 5 – August 17).

In [ ]:
import numpy as np

# Gare Centrale REM platform coordinates (Quai 1)
GCT_LAT = branch_stops[branch_stops.station_code == "GCT"].iloc[0].stop_lat
GCT_LON = branch_stops[branch_stops.station_code == "GCT"].iloc[0].stop_lon

RADIUS_M  = 500
CHUNK_SZ  = 500_000 # set to read data in 500k chunks
BIXI_COLS = [
    "STARTSTATIONLATITUDE", "STARTSTATIONLONGITUDE",
    "ENDSTATIONLATITUDE",   "ENDSTATIONLONGITUDE",
    "STARTTIMEMS",
]

def haversine_m(lats, lons, ref_lat, ref_lon):
    """
    This is the [Haversine formula](https://en.wikipedia.org/wiki/Haversine_formula), which returns a
    distance between two points on a sphere given latitude and longitude.
    """

    R = 6_371_000 # mean radius of the Earth
    phi1  = np.radians(lats)
    phi2  = np.radians(ref_lat)
    dphi  = np.radians(ref_lat - lats)
    dlam  = np.radians(ref_lon  - lons)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def load_bixi_window(csv_path):
    """Load trips in the closure window (Jul 5 - Aug 17) during peak hours in chunks."""

    chunks = []
    for chunk in pd.read_csv(csv_path, usecols=BIXI_COLS, chunksize=CHUNK_SZ):
        chunk["start_dt"] = (
            pd.to_datetime(chunk.STARTTIMEMS, unit="ms", utc=True)
            .dt.tz_convert("America/Montreal")
        )
        mmdd = chunk.start_dt.dt.month * 100 + chunk.start_dt.dt.day
        chunk = chunk[mmdd.between(705, 817)]
        hour = chunk.start_dt.dt.hour
        chunk = chunk[hour.between(7, 8) | hour.between(17, 18)]
        if len(chunk):
            chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=BIXI_COLS + ["start_dt"])

bixi_dir_2024 = next((BIXI_DIR / d) for d in sorted(p.name for p in BIXI_DIR.iterdir() if p.is_dir()) if "2024" in d)
bixi_dir_2025 = next((BIXI_DIR / d) for d in sorted(p.name for p in BIXI_DIR.iterdir() if p.is_dir()) if "2025" in d)

csv_2024 = next(bixi_dir_2024.glob("*.csv"))
csv_2025 = next(bixi_dir_2025.glob("*.csv"))

# load bixi '24
print("Loading 2024 Bixi data (closure window, peak hours)...")
bx24 = load_bixi_window(csv_2024)
print(f"  {len(bx24):,} trips loaded")

# load bixi '25
print("Loading 2025 Bixi data (closure window, peak hours)...")
bx25 = load_bixi_window(csv_2025)
print(f"  {len(bx25):,} trips loaded")

Loading 2024 Bixi data (closure window, peak hours)...
  742,696 trips loaded
Loading 2025 Bixi data (closure window, peak hours)...
  842,225 trips loaded


In [37]:
def near_gct(df):
    """Flag trips with a start or end station within RADIUS_M of Gare Centrale."""
    start_d = haversine_m(df.STARTSTATIONLATITUDE.values, df.STARTSTATIONLONGITUDE.values, GCT_LAT, GCT_LON)
    end_d   = haversine_m(df.ENDSTATIONLATITUDE.values,   df.ENDSTATIONLONGITUDE.values,   GCT_LAT, GCT_LON)
    return (start_d < RADIUS_M) | (end_d < RADIUS_M)

def summarise(df, label) -> pd.Series:
    """Split GCT-nearby data by morning/evening peaks and return a summary"""
    nearby = df[near_gct(df)].copy()
    hour   = nearby.start_dt.dt.hour
    am     = nearby[hour.between(7, 8)]
    pm     = nearby[hour.between(17, 18)]
    return pd.Series({
        "year":    label,
        "morning": len(am),
        "evening": len(pm),
        "total":   len(nearby),
    })

rows = [summarise(bx24, 2024), summarise(bx25, 2025)]
bixi_cmp = pd.DataFrame(rows).set_index("year")

# evaluate year-over-year change
delta = ((bixi_cmp.loc[2025] - bixi_cmp.loc[2024]) / bixi_cmp.loc[2024] * 100).round(1)

# organise and prettify cell output
print(f"Bixi trips within {RADIUS_M}m of Gare Centrale: Jul 5 - Aug 17 during peak hours\n")
print(f"{'':20} {'2024':>8} {'2025':>8} {'change':>8}")
print("─" * 48)
for col, label in [("morning", "Morning (7-9am)"), ("evening", "Evening (5-7pm)"), ("total", "Total")]:
    sign = "+" if delta[col] >= 0 else "" # type: ignore
    print(f"{label:20} {bixi_cmp.loc[2024, col]:>8,} {bixi_cmp.loc[2025, col]:>8,} {sign}{delta[col]:>6.1f}%")
print("─" * 48)
print(f"\nWindow: {(pd.Timestamp('2025-08-17') - pd.Timestamp('2025-07-05')).days + 1} days")
print(f"Radius: {RADIUS_M}m from REM Gare Centrale platform")

Bixi trips within 500m of Gare Centrale: Jul 5 - Aug 17 during peak hours

                         2024     2025   change
────────────────────────────────────────────────
Morning (7-9am)        21,025   23,653 +  12.5%
Evening (5-7pm)        26,273   29,883 +  13.7%
Total                  47,298   53,536 +  13.2%
────────────────────────────────────────────────

Window: 44 days
Radius: 500m from REM Gare Centrale platform


A 13% increase sounds like a signal, but Bixi ridership grows every year. We need to know whether near-GCT grew more or less than the network as a whole.
If citywide growth exceeds near-GCT growth, that relative underperformance is the actual signal: fewer REM arrivals feeding into the Gare Centrale node.

In [38]:
# establish city-wide growth patterns
citywide_24 = len(bx24)
citywide_25 = len(bx25)
citywide_delta = (citywide_25 - citywide_24) / citywide_24 * 100

# get delta and calculate residual
near_gct_delta = delta["total"]
residual = near_gct_delta - citywide_delta

print("Context: citywide Bixi peak-hour trips, same window")
print(f"  2024:            {citywide_24:>8,}")
print(f"  2025:            {citywide_25:>8,}")
print(f"  YoY change:      {citywide_delta:>+8.1f}%")
print()
print("Near-GCT vs network:")
print(f"  Citywide growth:     {citywide_delta:>+6.1f}%")
print(f"  Near-GCT growth:     {near_gct_delta:>+6.1f}%")
print(f"  Residual:            {residual:>+6.1f} pp")
print()
if abs(residual) < 3: # type: ignore
    print("Near-GCT growth is in line with city-wide growth. No meaningful signal.")
elif residual < 0: # type: ignore
    print(f"Near-GCT underperformed by {abs(residual):.1f} pp ...consistent with reduced REM arrivals at the island terminus.")
else:
    print(f"Near-GCT outperformed by {residual:.1f} pp ...possibly displaced commuters cycling the last mile from alternative drop-off points.")

Context: citywide Bixi peak-hour trips, same window
  2024:             742,696
  2025:             842,225
  YoY change:         +13.4%

Near-GCT vs network:
  Citywide growth:      +13.4%
  Near-GCT growth:      +13.2%
  Residual:              -0.2 pp

Near-GCT growth is in line with city-wide growth. No meaningful signal.



## 5. What the Data Can and Can't Tell Us

### Some conclusions

Based on the GTFS feeds:
- RTL boosted route 722 from 27 to 183 northbound weekday trips, which is a 578% increase! 
- The schedule delivered roughly comparable frequency to the REM (4-minute all-day headway, 2-minute peak headway). 
- The scheduled travel time was 35% longer, but the network existed (and it was dense).

The Bixi signal, by contrast, is flat. Near-Gare-Centrale peak-hour trips grew 13.2% year-over-year, which is practically indistinguishable from the 13.4% citywide growth rate. No detectable behavioural shift.

---

### Why the GTFS story is incomplete

GTFS captures what RTL *scheduled*, not what riders *experienced*. During a major rail closure, buses share the Champlain Bridge corridor with everyone who would have been driving instead. Bunching, delays, and unpredictable crowding are invisible in the feed. The 24-minute median is a schedule; on the ground it was probably noisier and sometimes worse.

The frequency boost on route 722 confirms RTL *anticipated* demand, which is a meaningful finding about planning intent. It says nothing about whether the buses ran full, whether riders showed up, or whether the service actually absorbed displaced ridership.

---

### Why the Bixi signal is silent

The null result isn't surprising. The REM Brossard branch serves tens of thousands of riders daily: even if a meaningful fraction shifted to cycling as part of a combined route, those additional trips would be absorbed into the tens of thousands of Bixi trips that already pass through the Gare Centrale node every summer weekday (tourism, errands, short commutes, etc.). 

The signal-to-noise ratio was probably never going to be good, but it was an interesting hypothesis to explore nonetheless. 

---

### What better data would look like

| Question | Better data |
|---|---|
| Did riders actually use route 722? | RTL automated passenger counts (APC) on route 722 |
| How many REM riders were displaced? | ARTM/REM fare gate counts at Brossard, before and during |
| Where did they go instead? | Anonymised trip plan data. Do routing requests from South Shore postal codes shift during the window? |
| Did driving increase? | Bridge toll or count data on the Champlain corridor |

### Ways to improve this analysis

- Is there other open data that might help respond to the problem questions? Do more research ahead of time. 
- More data visualization (mpl, seaborn, plotly, etc.), always!
- Mentally parse the data a bit more; "live in it" a bit more. Building a clearer mental image for myself always helps me work better.